# SNR Optimization Pipeline

## Overview

This notebook implements the initial stages of the **Py4DEEG** pipeline for system identification of cortical circuits using EEG recordings.

The pipeline performs the following steps:

1. Load and preprocess EEG recordings
2. Compute the Event Related Potential (ERP)
3. Perform source localization
4. Assess the Signal-to-Noise Ratio (SNR) in both electrode and source spaces

The analysis is implemented using **MNE-based workflows** built around MNE-Python.

---

## Folder structure

```
project_root/
│
├── SNR_Optimization.ipynb
│
├── functions_01/
│   ├── convertfuncs.py
│   ├── eegrelatedfuncs.py
│   ├── headtemplatefuncs.py
│   ├── leadfieldfuncs.py
│   ├── srclocalizationfuncs.py
│   ├── snrfuncs.py
│   └── setparm.py
|
├── data/
│   ├── raw/
│   └── processed_data/
│       └── P0
|           └── 01
|               └── S1
|                   ├── eeg
│                   |   ├── space-2mm-src.fif
│                   |   ├── space-5mm-src.fif
│                   |   ├── space-10mm-src.fif
│                   |   ├── space-15mm-src.fif
│                   |   ├── space-20mm-src.fif
│                   |   ├── template-trans.fif
│                   |   └── subject-epo.fif       # Here are the pre-processed epochs of each subject
|                   ├── leadfield
|                   |   └──  # Here are the forward models used in the study
|                   └── current
|                       └── # All source current and snr files are located here
|
└── subj_brains/
    └── subject_1/
        ├── bem/
        |   ├── 3shellbem.h5
        |   ├── brain.surf
        |   ├── inner_skull.surf
        |   ├── outer_skin.sufr
        |   └── outer_skull.surf
        |
        ├── label/
        |   ├── lh.HCPMMP1.surf
        |   └── rh.HCPMMP1.surf
        |
        ├── mri/
        |   ├── brain.mgz
        |   └── T1.mgz
        |
        └── surf/
            ├── lh.inflated
            ├── lh.pial
            ├── lh.sphere
            ├── lh.sphere (registration entries)
            ├── lh.white
            ├── rh.inflated
            ├── rh.pial
            ├── rh.sphere
            ├── rh.sphere (registration entries)
            └── rh.white
```

---

## Environment Setup

This notebook requires a Python environment with **MNE-based neuroimaging tools**.

Follow the official installation instructions from the MNE documentation:
https://mne.tools/stable/install/manual_install.html#manual-install

Recommended environment configuration:

* Python 3.10–3.11
* MNE-Python ≥ 1.5
* NumPy
* SciPy
* Matplotlib
* Pandas
* Seaborn

---

## Hardware Requirements

Due to the dataset size and source localization computations, the following is recommended:

* ≥ 8 GB RAM


## Imports

In [1]:
# -*- coding: utf-8 -*-

import h5py
import matplotlib
matplotlib.use('Qt5Agg') # makes plots appear in new window
import matplotlib.pyplot as plt
import sys
import os
import pandas  as pd
import seaborn as sns

sys.path.append("functions_01") 
from setparm               import set_parm
from eegrelatedfuncs       import eeg_preprocess
from convertfuncs          import convert_to_erp
from headtemplatefuncs     import load_head
from leadfieldfuncs        import prepare_source_space, prepare_leadfield
from srclocalizationfuncs  import source_localization, plot_stc, plot_same_source_different_methods,       \
                                  source_localization_split, plot_nmse_across_dists,                       \
                                  plot_split_nmse_participants
from snrfuncs              import measure_snr_el, measure_snr_src_distspaces,  measure_snr_src_repnum,     \
                                  measure_snr_src_multi_subject, plot_snr_el_multi_subject,                \
                                  load_all_snr_data, fit_lme_snr_model, measure_mse_between_splits,        \
                                  measure_snr_src_repnum_dist,  plot_combined_snr_hemispheric,             \
                                  plot_snr_vs_trials_sources

## Prepare subject path

This section prepares the subject paths and sets up the parm class with all the necessary definitions. The parm structure is the core of this pipeline, as it contains information about which subjects to include in the group analyses, filtering parameters, etc. It is located in /functions_01/setparm.py.

In [2]:
subject  = '01'   
protocol = 'P0'
session  = 'S1'
cwd      = os.getcwd()
subjpath = os.path.join(cwd,'data','raw_data',protocol,subject)
parm     = set_parm(subject, session, protocol)

Setting up parameters: started
Setting up parameters: completed


## Preprocess Raw EEG

In this section the raw EEG data are pre-processed. The code is included in this package for academic clarity, but for the sake of low memory consumption, the already processed files are included online.

In [7]:
eeg_preprocess(parm, False,False,False)  # (preica code, ica code, postica code)

## Convert to ERP

In this section, the ERP is measured for the subject 'subject'. This part generates the first figure pf the manuscript. Subject 01 was used for the figure.

In [8]:
convert_to_erp(parm)

Converting epochs to erp: started
Reading C:\Users\ioanniskyriazi\3_MainPython\Py4DEEG\data\processed_data\P0\06\S1\eeg\subject-epo.fif ...
    Found the data of interest:
        t =    -100.10 ...     200.20 ms
        0 CTF compensation matrices available
Not setting metadata
2000 matching events found
No baseline correction applied
0 projection items activated
Converting epochs to erp: completed


## Load Head & Brain

This section loads and prepares the head and brain files. No need to run it, the files are already in the processed data folders.

In [4]:
load_head(parm)

## Prepare source space

This section creates the 4 source spaces (5 mm, 10 mm, 15 mm and 20 mm). No need to run it, results are in processed_data\P0\{subject}\S1\leadfield.

In [12]:
prepare_source_space(parm)

## Prepare leadfield

In [ ]:
prepare_leadfield(parm)

## Perform Source Localization

Performs source localization for all participants, on all methods and all source spaces.

In [10]:
source_localization(parm)         # One group of 2000 epochs 

In [ ]:
source_localization_split(parm)   # Two groups of 1000 epochs each 

---
## Anything after this point requires all participant data to be generated

---

## Compare Source Localization Methods

Finds the source with the highest SNR for each participant and plots group total power and baseline per method and number of trials. Loads all participants, so execute this only when all files are generated.

In [1]:
plot_same_source_different_methods(parm) 

## Assess SNR in Sensor Space

For all participants creates the Monte Carlo curves for electrodes CP3, CPz and CP4. Essentially produces Figure 2 of the manuscript, as well as Supplementary Figure 1. Uses all participants, so only execute when all participant data are generated.

In [17]:
# File definition
csv_file_el = f"snr_{subject}_el.csv"  
measure_snr_el(parm,  output_csv=csv_file_el) 

In [3]:
plot_snr_el_multi_subject(parm)  # When all snr_{subject}_el.csv files have been generated, execute this. 

Plotting SNR, signal power, and noise power: started

SNR SUMMARY TABLE (Mean ± SD) - By Number of Trials
Trials      CP3 (Mean±SD)       Gain      Efficiency  CPz (Mean±SD)       Gain      Efficiency  CP4 (Mean±SD)       Gain      Efficiency  
------------------------------------------------------------------------------------------------------------------------
125         5.70±2.52                                 2.75±1.45                                 1.68±0.48                                 
250         8.65±4.16           +2.95     0.024       3.81±2.21           +1.06     0.008       1.87±0.68           +0.19     0.002       
500         11.99±6.53          +3.34     0.013       4.57±2.65           +0.76     0.003       2.12±0.83           +0.25     0.001        ← LOW-EFF
1000        15.13±7.98          +3.14     0.006       5.50±3.18           +0.93     0.002       2.81±1.30           +0.69     0.001        ← LOW-EFF
1500        18.64±10.47         +3.51     0.007       6.40

## Assess SNR in Source Space

Measures SNR in the source space for all participants. Also performs Wilcoxon signed rank tests. Needs all participant data to be generated first.

In [27]:
# File definition
csv_file_src_dist   = f"snr_{subject}_src_dist.csv"
csv_file_src        = f"snr_{subject}_src.csv"


method = 'eLORETA'  #Choose: sLORETA, eLORETA, MNE, dSPM

#measure_snr_src_repnum(parm, method, csv_file_src) 
#measure_snr_src_distspaces(parm,method, csv_file_src_dist)
#measure_snr_src_repnum_dist(parm, method,dist='5', output_csv ='snr_dSPM_5mm_source.csv')

measure_snr_src_multi_subject(parm, method,filter_s1_only=True)  #USED

## Compare Sensor & Source Spaces

When executed, it produces panels of C and D of Figure 3 of the manuscript. Uses all participant data, so execute when everything is generated.

In [30]:
plot_combined_snr_hemispheric(parm, method='eLORETA')

## Compare MSE between the 2 Splits

When executed, it produces Figure 4 of the Supplementary Material of the manuscript. Needs all participants to be generated.

In [ ]:
measure_mse_between_splits(parm)  # Splits (2 groups of 1000 trials each)

## Linear Mixed Effects Model

Needs all participants to be generated.

In [ ]:
# Define parameters
methods = ['eLORETA', 'sLORETA', 'MNE', 'dSPM']
dists   = [5, 10, 15, 20]

snr_df  = load_all_snr_data(parm, methods, dists)
result  = fit_lme_snr_model(snr_df)   
result

## Additional Source SNR Plot

In [4]:
plot_snr_vs_trials_sources(parm, dist_to_show=10, method_to_show='eLORETA', n_sources_to_show=10,  plot_mode='distance')   # 'snr_rank' or 'distance'

Subject: 01 | Method: eLORETA | Distance: 10 mm
Selected sources:
  src 128 [min, SNR=0.3]
  src 3033 [p11%, SNR=3.0]
  src 587 [p22%, SNR=5.7]
  src 687 [p33%, SNR=8.4]
  src 3531 [p44%, SNR=11.1]
  src 1921 [p55%, SNR=13.8]
  src 1503 [p66%, SNR=16.5]
  src 617 [p77%, SNR=19.2]
  src 1285 [p88%, SNR=21.8]
  src 1157 [max, SNR=24.5]
